# RoadWatch: cleaned-v3 YOLO11s training
Select a T4 GPU before running setup. Run cells in order. After a session reset, rerun sections 1–4 and then evaluate the saved checkpoint in section 6; skip section 5.


## 1. Install dependencies


In [ ]:
%pip install -q ultralytics==8.4.189


## 2. Mount Drive and extract the cleaned dataset
Upload `roadwatch_cleaned_v3.zip` to the root of My Drive first. Original `data/1` splits are not used.


In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import yaml

drive.mount("/content/drive")
ARCHIVE = Path("/content/drive/MyDrive/roadwatch_cleaned_v3.zip")
assert ARCHIVE.is_file(), "Upload roadwatch_cleaned_v3.zip to My Drive"
with zipfile.ZipFile(ARCHIVE) as archive:
    archive.extractall("/content")
DATA_ROOT = Path("/content/roadwatch_cleaned")
DATA_YAML = DATA_ROOT / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA_ROOT), "train": "train/images", "val": "valid/images",
    "test": "test/images", "nc": 1, "names": {0: "pothole"},
}))
RUNS_DIR = Path("/content/drive/MyDrive/RoadWatch_cleaned_training")
RUN_NAME = "pothole_yolo11s_cleaned_v3"


## 3. Verify cleaned-v3 splits
Counts and matching labels are checked here. Split provenance and similarity audits are documented in the dataset-cleaning notebook.


In [ ]:
EXPECTED = {"train": (3196, 8141), "valid": (375, 1097), "test": (369, 873)}
for split, (image_count, box_count) in EXPECTED.items():
    folder = DATA_ROOT / split
    images = [p for p in (folder / "images").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
    labels = list((folder / "labels").glob("*.txt"))
    assert len(images) == len(labels) == image_count, f"Wrong counts: {split}"
    assert {p.stem for p in images} == {p.stem for p in labels}, f"Unmatched labels: {split}"
    boxes = 0
    for label in labels:
        for line in label.read_text().splitlines():
            if not line.strip():
                continue
            fields = line.split()
            assert len(fields) == 5 and fields[0] == "0", f"Invalid label: {label}"
            x, y, w, h = map(float, fields[1:])
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1, label
            boxes += 1
    assert boxes == box_count, f"Wrong box count: {split}"
    print(f"{split}: {len(images)} images, {boxes} boxes")
print("Cleaned dataset ready")


## 4. Confirm GPU


In [ ]:
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU, then rerun setup"
DEVICE = 0
print("GPU:", torch.cuda.get_device_name(0))


## 5. Train from pretrained YOLO11s
Run only to start a new training experiment. Existing run directories are protected; use a new RUN_NAME for another experiment. Checkpoints are saved to Drive.


In [ ]:
from ultralytics import YOLO
assert not (RUNS_DIR / RUN_NAME).exists(), "Run already exists. Skip to section 6 to evaluate, or choose a new RUN_NAME."
model = YOLO("yolo11s.pt")
train_results = model.train(
    data=str(DATA_YAML), epochs=100, imgsz=640, batch=16,
    patience=20, device=DEVICE, seed=0,
    project=str(RUNS_DIR), name=RUN_NAME,
)
BEST_WEIGHTS = Path(model.trainer.save_dir) / "weights/best.pt"
print("Best weights:", BEST_WEIGHTS)


## 6. Evaluate the saved checkpoint on the held-out test split
After a reset, run sections 1–4 first. This cell does not train or change weights. Do not tune model settings against repeated test results.


In [ ]:
from ultralytics import YOLO
BEST_WEIGHTS = RUNS_DIR / RUN_NAME / "weights/best.pt"
assert BEST_WEIGHTS.is_file(), "Saved checkpoint not found in Drive"
best_model = YOLO(str(BEST_WEIGHTS))
test_metrics = best_model.val(
    data=str(DATA_YAML), split="test", imgsz=640, device=DEVICE,
    project=str(RUNS_DIR / RUN_NAME), name="test_evaluation", plots=True,
)
print(f"Test precision: {test_metrics.box.mp:.4f}")
print(f"Test recall: {test_metrics.box.mr:.4f}")
print(f"Test mAP50: {test_metrics.box.map50:.4f}")
print(f"Test mAP50-95: {test_metrics.box.map:.4f}")


## 7. Download the checkpoint
The checkpoint already lives in Drive. Download it as one file and place it at `models/best.pt` in your local repository. Do not extract it.


In [ ]:
from google.colab import files
files.download(str(BEST_WEIGHTS))
